# Reviewer: RF spatial structure vs grating tuning

Does the shape of the sphere RF predict the drifting-grating tuning of the same neurons?

- **A. Orientation**: a red centre with blue flanks on either side should prefer gratings whose bars are
  parallel to the flanks, i.e. whose wave vector joins the flanks (flanks above and below the centre:
  horizontal bars).
- **B. Direction**: RFs with a single or unequal flank (asymmetric) should be more direction selective, and
  may prefer motion along the asymmetry axis.

Same 6 sessions and cells as `fig_reviewer_gratings` (its `grating_tuning_cells.csv` is the grating input).

**RFs**: spike RFs (`neurons_df_spks.pickle`, as in the manuscript), closed loop, mean of the 5 folds, at
the depth of the RF peak: a 16 × 24 map of 5° pixels (elevation -40 to 40°, azimuth 0 to 120°). Only cells
with a significant RF (`rf_sig`, peak > 6 SD of the ipsilateral RF) whose peak is at least `MAX_DIST`
pixels from every edge (so the surround is not truncated, which would fake asymmetry).
Shape metrics (`figure_utils/rf_shape.py`), in units of the ipsilateral RF noise (z), from circular moments
of the negative surround in a 4-pixel (20°) disk around the peak:
- *flank axis*: direction joining the flanks (axial moment); *anisotropy*: 0 ring, 1 flanks on one line;
- *asymmetry*: vector moment, 0 for symmetric flanks, 1 for a single flank; *asymmetry direction*:
  towards the deeper flank;
- *flank z*: deepest surround value on each side of the peak along the axis; *flanked* = deepest side
  ≥ `FLANK_Z`, *bilateral* = both sides ≥ `FLANK_Z`;
- *centre elongation*: long axis of the positive centre (classic predictor of orientation preference);
- a 2D Gabor fit (orientation, SF, phase) as a parametric cross-check.

**Grating convention**, from the stimulus workflow (`vis-stim-2d/run_SFTF.bonsai`, identical in the version
of 2023-09-11, `fa05b40`, used for these recordings) and BonVision:
- luminance = sin(2π(SF · (cos A, sin A) · x + TF · (t - t_trial) - phase)), so `Angle` A is the wave
  vector, the bars run at A + 90° and, as the phase grows with time, drift towards A + 180°;
- the grating is drawn in an `OrthographicView` (x 0 to 120°, y ±50°) wrapped on the sphere by
  `SphereMapping` (`CreateSphereGrid`): x is azimuth from the front, increasing to the right, and y is
  elevation, upwards. The RF reconstruction (`stimulus_reconstruction.draw_spheres`) uses the same axes
  (compass azimuth from the front, increasing to the right; elevation up), so angles in the RF map and
  grating angles are in the same frame: texture y = elevation (`CONVENTIONS` first entry).
So `pref_ori` (fitted `alpha0` mod 180°) is the preferred wave-vector orientation, directly comparable to the
flank axis, and the preferred motion direction is `pref_dir` + 180°. A = 0 is vertical bars drifting
towards lower azimuth (backward to forward); A = 90° is horizontal bars drifting down. The mirrored
convention is kept in the tests as a control.

**Statistics**: permutation of the grating preferences among the selected cells within each session
(keeps the population distributions of both measures, e.g. cardinal biases), `N_PERM` permutations;
correlations with the hierarchical bootstrap over mice and sessions, as in `fig_reviewer_gratings`.
Exploratory: nothing is exported to the manuscript stats.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import contextlib
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import flexiznam as flz

from cottage_analysis.analysis import common_utils
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM
from v1_depth_map.figure_utils import rf_shape
from v1_depth_map.figure_utils.rf_distance import load_session_neurons
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
SESSIONS = [
    "PZAH10.2d_S20230920",
    "PZAH10.2d_S20230922",
    "PZAH10.2f_S20230817",
    "PZAH10.2f_S20230912",
    "PZAH10.2f_S20230914",
    "PZAH10.2f_S20230924",
]
RF_FILE = "neurons_df_spks.pickle"  # spike RFs, as in the manuscript
MAX_DIST = rf_shape.MAX_DIST  # surround radius (pixels) and minimum distance of the peak to the edges
HALVES = [[0, 1], [2, 3, 4]]  # RF folds of the split-half reliability
FLANK_Z = 2  # flank depth (z) for "flanked" / "bilateral"
ASYM_THR = 0.3  # B, signed tests: asymmetric RFs only
GABOR_RSQ = 0.5  # Gabor cross-check: fits explaining at least this much of the map
CONVENTIONS = {"texture y = elevation": 1, "texture y = -elevation": -1}  # map angle = sign * grating angle
N_PERM = 10000
N_BOOTS = 20000
REBUILD = False  # True: recompute the RF shape table instead of reading the cache

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_rf_sftf")
_, GRATINGS_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_gratings")
CACHE = SAVE_ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Saving to {SAVE_ROOT}")

## RF shape metrics

One row per `iscell` ROI. Maps (in z) of all significant RFs, centred or not, are kept for the example
figures. Cached in `CACHE`.

In [ ]:
def session_rf_shape(session):
    neurons_df, _ = load_session_neurons(flm, session, rf_file=RF_FILE)
    coef = np.stack(neurons_df.rf_coef_closedloop.values)  # (n_rois, k_folds, n_features)
    vol = rf_shape.rf_maps(coef)
    vol_ipsi = rf_shape.rf_maps(np.stack(neurons_df.rf_coef_ipsi_closedloop.values))
    halves = [rf_shape.rf_maps(coef, h) for h in HALVES]
    peak = rf_shape.peak_index(vol)
    centred = rf_shape.is_centred(peak, max_dist=MAX_DIST)
    rows, maps = [], {}
    for roi in np.flatnonzero(neurons_df.iscell.values.astype(bool)):
        d, iy, ix = peak[roi]
        row = dict(
            session=session, roi=roi, rf_sig=bool(neurons_df.rf_sig.iloc[roi]),
            rf_azi=neurons_df.rf_azi.iloc[roi], rf_ele=neurons_df.rf_ele.iloc[roi],
            rf_idepth=d, centred=bool(centred[roi]),
        )
        noise = np.nanstd(vol_ipsi[roi, d])  # ipsilateral RF at the same depth
        if row["rf_sig"] and d >= 0:
            maps[(session, roi)] = dict(img=vol[roi, d] / noise, iy=iy, ix=ix)
        if centred[roi]:
            m = rf_shape.flank_metrics(vol[roi, d], iy, ix, noise, radius=MAX_DIST)
            row.update({k: v for k, v in m.items() if k != "flank_z"})
            row["flank_z0"], row["flank_z1"] = m["flank_z"]
            for h, half in enumerate(halves):
                mh = rf_shape.flank_metrics(half[roi, d], iy, ix, noise, radius=MAX_DIST)
                for k in ["axis", "asym", "asym_dir", "oriented_z", "elong_axis"]:
                    row[f"{k}_h{h}"] = mh[k]
            if row["rf_sig"]:
                g = rf_shape.fit_gabor(vol[roi, d], iy, ix)
                row.update({f"gabor_{k}": v for k, v in g.items()})
        rows.append(row)
    return pd.DataFrame(rows), maps


path = CACHE / "rf_shape.pickle"
if path.exists() and not REBUILD:
    rf, rf_maps = pd.read_pickle(path).values()
else:
    out = [session_rf_shape(s) for s in SESSIONS]
    rf = pd.concat([o[0] for o in out], ignore_index=True)
    rf_maps = {k: v for o in out for k, v in o[1].items()}
    pd.to_pickle(dict(table=rf, maps=rf_maps), path)
print(f"{len(rf)} cells, {rf.rf_sig.sum()} with a significant RF, {(rf.rf_sig & rf.centred).sum()} of them centred")

In [ ]:
grating_df = pd.read_csv(GRATINGS_ROOT / "grating_tuning_cells.csv")
cells = grating_df.merge(rf, on=["session", "roi"], how="left", validate="one_to_one")
assert cells.rf_sig.notna().all(), "grating cells missing from the RF table"
cells["mouse"] = cells.session.str.split("_").str[0]
cells["flank_max"] = cells[["flank_z0", "flank_z1"]].max(axis=1)
cells["flank_min"] = cells[["flank_z0", "flank_z1"]].min(axis=1)
cells["shape_ok"] = cells.rf_sig & cells.centred
cells["flanked"] = cells.shape_ok & (cells.flank_max >= FLANK_Z)
cells["bilateral"] = cells.shape_ok & (cells.flank_min >= FLANK_Z)
cells["motion_dir"] = (cells.pref_dir + 180) % 360  # bars drift towards Angle + 180
cells["fit_motion_dir"] = (cells.fit_dir + 180) % 360
cells["flow_dir"] = rf_shape.flow_direction(cells.rf_azi, cells.rf_ele)

funnel = cells.groupby("session").agg(
    iscell=("roi", "size"),
    rf_sig=("rf_sig", "sum"),
    rf_sig_centred=("shape_ok", "sum"),
    flanked=("flanked", "sum"),
    bilateral=("bilateral", "sum"),
    ori_selective=("ori_selective", lambda s: (s & cells.loc[s.index, "shape_ok"]).sum()),
    dir_selective=("dir_selective", lambda s: (s & cells.loc[s.index, "shape_ok"]).sum()),
    sftf_tuned=("sftf_tuned", lambda s: (s & cells.loc[s.index, "shape_ok"]).sum()),
)
funnel.loc["total"] = funnel.sum()
funnel  # last three columns: among rf_sig_centred

## Example RFs

RF maps (z, at the RF peak depth) of the most clearly bilateral and the most asymmetric cells with a grating
preference. Green line: flank axis; green arrow: asymmetry direction (length ∝ asymmetry); black dashed:
preferred grating wave vector (`pref_ori`), black arrow: preferred motion direction, both drawn with
texture y = elevation.

In [ ]:
EXTENT = [0, rf_shape.FRAME_SHAPE[1] * rf_shape.RES, -rf_shape.FRAME_SHAPE[0] * rf_shape.RES / 2,
          rf_shape.FRAME_SHAPE[0] * rf_shape.RES / 2]


def pix_to_deg(iy, ix):
    return (ix + 0.5) * rf_shape.RES + EXTENT[0], (iy + 0.5) * rf_shape.RES + EXTENT[2]


def plot_rf(ax, row, length=15):
    m = rf_maps[(row.session, row.roi)]
    lim = np.nanmax(np.abs(m["img"]))
    ax.imshow(m["img"], origin="lower", extent=EXTENT, cmap="RdBu_r", vmin=-lim, vmax=lim)
    x0, y0 = pix_to_deg(m["iy"], m["ix"])
    th = np.deg2rad(row.axis)
    ax.plot([x0 - length * np.cos(th), x0 + length * np.cos(th)],
            [y0 - length * np.sin(th), y0 + length * np.sin(th)], color="#1baf7a", lw=1)
    th = np.deg2rad(row.asym_dir)
    ax.annotate("", (x0 + 2 * length * row.asym * np.cos(th), y0 + 2 * length * row.asym * np.sin(th)),
                (x0, y0), arrowprops=dict(arrowstyle="->", color="#1baf7a", lw=1))
    th = np.deg2rad(row.pref_ori)
    ax.plot([x0 - length * np.cos(th), x0 + length * np.cos(th)],
            [y0 - length * np.sin(th), y0 + length * np.sin(th)], color="k", lw=0.8, ls="--")
    th = np.deg2rad(row.motion_dir)
    ax.annotate("", (x0 + length * np.cos(th), y0 + length * np.sin(th)), (x0, y0),
                arrowprops=dict(arrowstyle="->", color="k", lw=0.8))
    ax.set_xlim(x0 - 30, x0 + 30)
    ax.set_ylim(y0 - 30, y0 + 30)
    ax.set_title(f"{row.session[:9]} {row.roi}\nflanks {row.flank_z0:.1f}/{row.flank_z1:.1f} z, asym "
                 f"{row.asym:.2f}\nOSI {row.osi:.2f} dsi {row.fit_dsi:.2f}", fontsize=5)
    ax.tick_params(labelsize=5)


responsive_rf = cells[cells.shape_ok & cells.responsive]
ex_bilateral = responsive_rf[responsive_rf.ori_selective].nlargest(6, "flank_min")
ex_asym = responsive_rf[responsive_rf.flanked & responsive_rf.dir_selective].assign(
    score=lambda d: d.asym * d.flank_max).nlargest(6, "score")
fig, axes = plt.subplots(2, 6, figsize=(18 * CM, 7.5 * CM))
for ax, (_, row) in zip(axes[0], ex_bilateral.iterrows()):
    plot_rf(ax, row)
for ax, (_, row) in zip(axes[1], ex_asym.iterrows()):
    plot_rf(ax, row)
axes[0, 0].set_ylabel("Bilateral flanks\n(ori. selective)\nElevation (deg)", fontsize=6)
axes[1, 0].set_ylabel("Asymmetric\n(dir. selective)\nElevation (deg)", fontsize=6)
fig.tight_layout()
style.savefig(SAVE_ROOT / "rf_examples.pdf", fig=fig, bbox_inches="tight")

## All significant RFs with grating selectivity

Every cell with a significant RF (`rf_sig`, centred or not) that is orientation or direction selective: whole
RF map (z, azimuth 0-120°, elevation -40-40°) at the RF peak depth, with the preferred bar orientation
(`pref_ori` + 90°, dashed, only for fitted OSI ≥ `GALLERY_OSI`) and preferred motion direction (`pref_dir` + 180°,
arrow centred on the RF peak, length ∝ fitted `dsi`, only for `dsi` ≥ `GALLERY_DSI`). Cells with OSI ≥
`GALLERY_OSI` first, then those with OSI < `GALLERY_OSI` and `dsi` ≥ `GALLERY_DSI`, then the rest, each group
sorted by preferred motion direction. `pref_ori` and `pref_dir` both come from the fitted `alpha0`, so the line
and arrow are orthogonal.

In [ ]:
GALLERY_COLOR = "#1baf7a"
ARROW_GAIN = 100  # arrow length (deg) per unit dsi
GALLERY_OSI = 1 / 3  # dashed bar-orientation line only from this fitted OSI (as ori_selective)
GALLERY_DSI = 0.6  # direction arrow only from this fitted dsi (as dir_selective)


def plot_rf_grating(ax, row, length=30):
    # same colours as figure 4 (rf_plots.plot_rf): bwr, symmetric limits at the RF maximum
    m = rf_maps[(row.session, row.roi)]
    lim = np.nanmax(m["img"])
    ax.imshow(m["img"], origin="lower", extent=EXTENT, cmap="bwr", vmin=-lim, vmax=lim)
    x0, y0 = pix_to_deg(m["iy"], m["ix"])
    if row.osi >= GALLERY_OSI:
        th = np.deg2rad(row.pref_ori + 90)  # bars are orthogonal to the wave vector
        ax.plot([x0 - length * np.cos(th), x0 + length * np.cos(th)],
                [y0 - length * np.sin(th), y0 + length * np.sin(th)], color=GALLERY_COLOR, lw=1.3, ls="--")
    if row.fit_dsi >= GALLERY_DSI:  # motion direction: arrow centred on the RF peak, length ∝ dsi
        th = np.deg2rad(row.motion_dir)
        half = ARROW_GAIN * row.fit_dsi / 2
        ax.annotate("", (x0 + half * np.cos(th), y0 + half * np.sin(th)),
                    (x0 - half * np.cos(th), y0 - half * np.sin(th)),
                    arrowprops=dict(arrowstyle="-|>", color=GALLERY_COLOR, lw=1, mutation_scale=6,
                                    shrinkA=0, shrinkB=0),
                    annotation_clip=False)  # still draw arrows whose head is outside the map
    ax.set_xlim(EXTENT[:2])
    ax.set_ylim(EXTENT[2:])
    sel = "O" * row.ori_selective + "D" * row.dir_selective
    ax.set_title(f"{row.motion_dir:.0f}° {row.session[:9]} {row.roi} ({sel})\n"
                 f"OSI {row.osi:.2f} dsi {row.fit_dsi:.2f}", fontsize=4)
    ax.set_xticks([])
    ax.set_yticks([])


gallery = (
    cells[cells.rf_sig & (cells.ori_selective | cells.dir_selective)]
    # groups: OSI >= GALLERY_OSI, then OSI < GALLERY_OSI with dsi >= GALLERY_DSI, then the rest
    .assign(low_osi=lambda d: d.osi < GALLERY_OSI, low_dsi=lambda d: d.low_osi & (d.fit_dsi < GALLERY_DSI))
    .sort_values(["low_osi", "low_dsi", "motion_dir", "session", "roi"])
)
n_cols = 10
n_rows = int(np.ceil(len(gallery) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18 * CM, 1.6 * CM * n_rows), squeeze=False)
for ax, (_, row) in zip(axes.flat, gallery.iterrows()):
    plot_rf_grating(ax, row)
for ax in axes.flat[len(gallery):]:
    ax.axis("off")
fig.suptitle(f"{len(gallery)} cells: significant RF and orientation (O) or direction (D) selective; OSI ≥ "
             f"{GALLERY_OSI:.2f} first ({(~gallery.low_osi).sum()} cells), then OSI < {GALLERY_OSI:.2f} with dsi ≥ {GALLERY_DSI} "
             f"({(gallery.low_osi & ~gallery.low_dsi).sum()} cells), each group sorted by preferred motion "
             "direction; whole RF map (azimuth 0-120°, elevation -40-40°)", fontsize=6)
fig.tight_layout(rect=(0, 0, 1, 1 - 0.6 / (1.6 * n_rows)))  # room for the suptitle
style.savefig(SAVE_ROOT / "rf_gallery_ori_dir_selective.pdf", fig=fig, bbox_inches="tight", dpi=600)

## Consistency of the shape metrics across folds

The same metrics on RFs averaged over folds 1-2 and over folds 3-5 (`HALVES`), for centred significant RFs.
**These halves are not independent**: each fold's RF is fitted on the other 4 folds' data, so the two
averages share most of their training data and the agreement below is an upper bound. It only shows that the
metrics are not dominated by fold-to-fold fitting noise; an independent split would need a refit on
separate halves of the trials.

In [ ]:
ok = cells[cells.shape_ok]
rel = {}
for name, sel in [("all centred RFs", ok), (f"flanked (z ≥ {FLANK_Z})", ok[ok.flanked])]:
    d_axis = np.abs(rf_shape.axial_diff(sel.axis_h0, sel.axis_h1))
    d_elong = np.abs(rf_shape.axial_diff(sel.elong_axis_h0, sel.elong_axis_h1))
    d_asym = np.abs(rf_shape.axial_diff(sel.asym_dir_h0, sel.asym_dir_h1, period=360))
    rel[name] = dict(
        n=len(sel),
        flank_axis_median_diff=np.nanmedian(d_axis),
        elong_axis_median_diff=np.nanmedian(d_elong),
        asym_rho=stats.spearmanr(sel.asym_h0, sel.asym_h1, nan_policy="omit")[0],
        asym_dir_median_diff=np.nanmedian(d_asym),
    )
rel = pd.DataFrame(rel).T
print("chance: median axis difference 45 deg, median direction difference 90 deg")
rel.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15 * CM, 4.5 * CM))
fl = ok[ok.flanked]
axes[0].hist(np.abs(rf_shape.axial_diff(fl.axis_h0, fl.axis_h1)), bins=18, range=(0, 90), color="k")
axes[0].set_xlabel("|Δ flank axis| between halves (deg)")
axes[1].hist(np.abs(rf_shape.axial_diff(ok.elong_axis_h0, ok.elong_axis_h1)), bins=18, range=(0, 90), color="k")
axes[1].set_xlabel("|Δ centre long axis| between halves (deg)")
axes[2].scatter(fl.asym_h0, fl.asym_h1, s=4, color="k", linewidths=0)
axes[2].set_xlabel("Asymmetry, folds 1-2")
axes[2].set_ylabel("Asymmetry, folds 3-5")
axes[0].set_ylabel("Number of cells")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "rf_shape_reliability.pdf", fig=fig, bbox_inches="tight")

## Statistics helpers

- `axial_test`: weighted mean of cos 2Δ between an RF axis and the grating wave-vector orientation
  (1: all aligned, 0: unrelated, -1: all orthogonal), one-sided permutation p for alignment.
- `direction_test`: mean cos Δ between two directions, two-tailed permutation p (more aligned or more
  opposite than the null, which is not centred on 0 when both sets of directions are clustered).
- `bootstrap_spearman`: Spearman ρ with the hierarchical bootstrap over mice and sessions.

In [ ]:
rng = np.random.default_rng(0)


def permutations(groups, n=N_PERM):
    # (n, n_cells) index arrays permuting the cells within each group
    groups = np.asarray(groups)
    perm = np.tile(np.arange(len(groups)), (n, 1))
    for g in np.unique(groups):
        idx = np.flatnonzero(groups == g)
        perm[:, idx] = idx[rng.permuted(np.tile(np.arange(len(idx)), (n, 1)), axis=1)]
    return perm


def axial_test(rf_axis, grating_ori, groups, weights=None):
    rf_axis, grating_ori = np.asarray(rf_axis, float), np.asarray(grating_ori, float)
    w = np.ones(len(rf_axis)) if weights is None else np.asarray(weights, float)
    stat = lambda g: (w * np.cos(2 * np.deg2rad(rf_axis - g))).sum(axis=-1) / w.sum()
    obs, null = stat(grating_ori), stat(grating_ori[permutations(groups)])
    return dict(n=len(rf_axis), stat=obs, p=(1 + (null >= obs).sum()) / (len(null) + 1), null=null)


def direction_test(dir_a, dir_b, groups):
    dir_a, dir_b = np.asarray(dir_a, float), np.asarray(dir_b, float)
    stat = lambda b: np.cos(np.deg2rad(dir_a - b)).mean(axis=-1)
    obs, null = stat(dir_b), stat(dir_b[permutations(groups)])
    # two-tailed on the null itself, which is not centred on 0 when both directions are clustered
    tail = min((null >= obs).sum(), (null <= obs).sum())
    return dict(n=len(dir_a), stat=obs, p=min(1, 2 * (1 + tail) / (len(null) + 1)), null=null)


def bootstrap_spearman(x, y, sessions):
    df = pd.DataFrame(dict(x=np.asarray(x, float), y=np.asarray(y, float), session=np.asarray(sessions)))
    df = df[np.isfinite(df.x) & np.isfinite(df.y)].reset_index(drop=True)
    with contextlib.redirect_stderr(io.StringIO()):  # silence the tqdm progress bar
        rho, dist = common_utils.hierarchical_bootstrap_stats(
            df, N_BOOTS, ["x"], ["mouse", "session"], ycol=["y"], correlation=True, plot=False
        )
    return dict(
        n=len(df),
        rho=float(rho[0]),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
    )


def fmt(t):
    return f"n = {t['n']}, stat = {t['stat']:.3f}, p = {t['p']:.3g}"

## A. Orientation: RF axes vs preferred grating orientation

Orientation-selective cells (`ori_selective`: SF/TF tuned and fitted OSI > 1/3) with a centred
significant RF.
- **A1 flank axis** vs `pref_ori`, each cell weighted by the strength of its oriented flank structure
  (`oriented_z`), so that no flank threshold is needed and ring-like or flat surrounds count little.
- **A2 centre long axis**: an RF centre elongated along the bars predicts the wave vector orthogonal to it
  (`elong_axis` + 90°), weighted by `elongation`.
- **A3 Gabor**: wave-vector orientation of the Gabor fit (R² ≥ `GABOR_RSQ`), unweighted.
- **A4 SF/TF fit**: A1 with the fitted orientation (`fit_ori`) of SF/TF-tuned cells.

Each under both texture-y conventions; the stat is the weighted mean cos 2Δ (null 0).

In [ ]:
selA = cells[cells.shape_ok & cells.ori_selective & (cells.oriented_z > 0)].reset_index(drop=True)
selA_fit = cells[cells.shape_ok & cells.sftf_tuned & (cells.oriented_z > 0)].reset_index(drop=True)
selA_gabor = cells[cells.shape_ok & cells.ori_selective & (cells.gabor_rsq >= GABOR_RSQ)].reset_index(drop=True)
testsA = {}
for conv, sign in CONVENTIONS.items():
    testsA[("A1 flank axis", conv)] = axial_test(selA.axis, sign * selA.pref_ori, selA.session, selA.oriented_z)
    testsA[("A2 centre long axis", conv)] = axial_test(
        selA.elong_axis + 90, sign * selA.pref_ori, selA.session, selA.elongation)
    testsA[("A3 Gabor", conv)] = axial_test(selA_gabor.gabor_theta, sign * selA_gabor.pref_ori, selA_gabor.session)
    testsA[("A4 flank axis, SF/TF fit", conv)] = axial_test(
        selA_fit.axis, sign * selA_fit.fit_ori, selA_fit.session, selA_fit.oriented_z)
for (name, conv), t in testsA.items():
    print(f"{name:26s} {conv:22s} {fmt(t)}")

### A5. The specific hypothesis: bilateral flanks and horizontal gratings

Cardinal comparisons only, so independent of the texture-y convention. Among orientation-selective cells
with a centred significant RF: is the preferred orientation horizontal bars (`pref_ori` within 22.5° of 90°)
more often for cells with bilateral flanks than for the others, and is it specifically for those whose
flanks are above and below the centre (flank axis within 22.5° of 90°)? The same for vertical bars
(`pref_ori` within 22.5° of 0°), which flanks above and below would predict if the flanks were end
inhibition rather than side bands. Fisher exact tests.

In [ ]:
ori_rf = cells[cells.shape_ok & cells.ori_selective].copy()
ori_rf["horizontal_bars"] = np.abs(rf_shape.axial_diff(ori_rf.pref_ori, 90)) < 22.5
ori_rf["vertical_flanks"] = ori_rf.bilateral & (np.abs(rf_shape.axial_diff(ori_rf.axis, 90)) < 22.5)
ori_rf["horizontal_flanks"] = ori_rf.bilateral & (np.abs(rf_shape.axial_diff(ori_rf.axis, 0)) < 22.5)
testsA5 = {}
ori_rf["vertical_bars"] = np.abs(rf_shape.axial_diff(ori_rf.pref_ori, 0)) < 22.5
for col in ["bilateral", "vertical_flanks", "horizontal_flanks"]:
    for bars in ["horizontal_bars", "vertical_bars"]:
        table = pd.crosstab(ori_rf[col], ori_rf[bars]).reindex(index=[False, True], columns=[False, True], fill_value=0)
        testsA5[(col, bars)] = dict(table=table, p=stats.fisher_exact(table.values)[1],
                                    frac=table.div(table.sum(axis=1), axis=0)[True].round(2).to_dict())
        print(f"{col:17s} -> {bars:15s}: fraction (other cells, these cells) = "
              f"{list(testsA5[(col, bars)]['frac'].values())}, n = {list(table.sum(axis=1))}, "
              f"Fisher p = {testsA5[(col, bars)]['p']:.3g}")

# flank axis bin x preferred orientation bin, bilateral cells (bins centred on the stimulus orientations)
bins = lambda a: ((a + 22.5) % 180 // 45 * 45).astype(int)
bil = ori_rf[ori_rf.bilateral]
pd.crosstab(bins(bil.axis).rename("flank axis"), bins(bil.pref_ori).rename("pref_ori (wave vector)"))

## B. Direction: RF asymmetry vs direction selectivity

- **B1** asymmetry vs fitted `fit_dsi` (SF/TF-tuned cells), flanked cells
  with a centred significant RF (asymmetry is undefined without a surround). Hierarchical bootstrap.
- **B2** asymmetry of direction-selective vs other SF/TF-tuned flanked cells (Mann-Whitney).
- **B3** signed: for direction-selective cells with an asymmetric RF (asym ≥ `ASYM_THR`), does the preferred
  motion direction point towards (cos Δ > 0) or away from (< 0) the deeper flank? Both conventions.
- **B4** optic-flow and position controls: during the RF mapping the spheres move away from the focus of
  expansion, and the RF is fitted at a fixed lag, so a direction-selective cell could show an RF smeared
  along the local flow. Over these RFs the flow direction is nearly the same for every cell (towards
  increasing azimuth), so a shuffle across cells cannot separate flow from RF position. Instead: V-test of
  the asymmetry direction against each cell's flow direction (null: uniform directions,
  V·sqrt(2n) ~ N(0, 1)), and Spearman correlations of the asymmetry vector (cos, sin) with RF azimuth and
  elevation, which would show asymmetry set by where the RF is rather than by the cell.

In [ ]:
flk = cells[cells.flanked].reset_index(drop=True)
testsB = {
    "asym vs fit_dsi (SF/TF tuned)": bootstrap_spearman(
        flk.asym[flk.sftf_tuned], flk.fit_dsi[flk.sftf_tuned], flk.session[flk.sftf_tuned]),
    "|sin Gabor phase| vs fit_dsi (SF/TF tuned)": bootstrap_spearman(
        *(lambda d: (np.abs(np.sin(d.gabor_phase)), d.fit_dsi, d.session))(
            cells[cells.shape_ok & cells.sftf_tuned & (cells.gabor_rsq >= GABOR_RSQ)])),
}
for name, st in testsB.items():
    print(f"{name:45s} n = {st['n']}, rho = {st['rho']:.2f} [{st['ci'][0]:.2f}, {st['ci'][1]:.2f}], p = {st['p']:.3g}")

resp_flk = flk[flk.sftf_tuned]  # dir_selective is a subset of the SF/TF-tuned cells
a, b = resp_flk.asym[resp_flk.dir_selective], resp_flk.asym[~resp_flk.dir_selective]
testB2 = dict(n=(len(a), len(b)), median=(a.median(), b.median()), p=stats.mannwhitneyu(a, b).pvalue)
print(f"B2 asym, direction selective vs not: n = {testB2['n']}, medians = {np.round(testB2['median'], 2)}, "
      f"p = {testB2['p']:.3g}")

In [ ]:
selB3 = flk[flk.dir_selective & (flk.asym >= ASYM_THR)].reset_index(drop=True)
selB4 = flk[flk.asym >= ASYM_THR].reset_index(drop=True)
selB4_dir = cells[cells.shape_ok & cells.dir_selective].reset_index(drop=True)
testsB3 = {
    conv: direction_test(selB3.asym_dir, (sign * selB3.motion_dir) % 360, selB3.session)
    for conv, sign in CONVENTIONS.items()
}
def v_test(dirs, reference):
    # V = mean cos(dirs - reference); two-sided p against uniform directions
    v = np.cos(np.deg2rad(np.asarray(dirs) - np.asarray(reference))).mean()
    return dict(n=len(dirs), stat=v, p=2 * stats.norm.sf(abs(v) * np.sqrt(2 * len(dirs))))


flow = selB4.flow_dir.values
print(f"flow direction over these RFs: circular mean {np.rad2deg(np.angle(np.exp(1j * np.deg2rad(flow)).mean())) % 360:.0f} deg, "
      f"resultant length {np.abs(np.exp(1j * np.deg2rad(flow)).mean()):.2f}")
testsB4 = {"asymmetry direction vs flow": v_test(selB4.asym_dir, selB4.flow_dir)}
for conv, sign in CONVENTIONS.items():
    testsB4[f"grating motion vs flow, {conv}"] = v_test((sign * selB4_dir.motion_dir) % 360, selB4_dir.flow_dir)
th = np.deg2rad(selB4.asym_dir)
position = {(pos, comp): stats.spearmanr(selB4[pos], f(th))
            for pos in ["rf_azi", "rf_ele"] for comp, f in [("cos", np.cos), ("sin", np.sin)]}
for conv, t in testsB3.items():
    print(f"B3 asymmetry direction vs motion, {conv:22s} {fmt(t)}")
for name, t in testsB4.items():
    print(f"B4 {name:45s} {fmt(t)}")
for (pos, comp), r in position.items():
    print(f"B4 asymmetry {comp} vs {pos}: rho = {r[0]:.2f}, p = {r[1]:.2g}, n = {len(selB4)}")

## Summary figure

- **A, B**: flank axis - preferred wave-vector orientation (A1), each convention; bars weighted by
  `oriented_z`, stat and permutation p in the title.
- **C**: centre long axis + 90° - preferred orientation (A2), texture y = elevation.
- **D**: asymmetry vs fitted DSI (B1, SF/TF-tuned flanked cells).
- **E**: preferred motion direction - asymmetry direction, direction-selective asymmetric cells (B3),
  texture y = elevation.
- **F**: asymmetry direction vs RF azimuth (B4 position control): the deeper flank points to low azimuth
  (cos < 0) for RFs at large azimuth.

In [ ]:
def half_polar(ax, delta, weights, title):
    # weighted histogram of axial differences in [-90, 90), drawn on a half circle
    counts, edges = np.histogram(delta, bins=12, range=(-90, 90), weights=weights)
    ax.bar(np.deg2rad(edges[:-1] + 7.5), counts, width=np.deg2rad(15) * 0.95, color="k", lw=0)
    ax.set_thetamin(-90)
    ax.set_thetamax(90)
    ax.set_xticks(np.deg2rad([-90, -45, 0, 45, 90]))
    ax.yaxis.set_major_locator(plt.MaxNLocator(2))
    ax.tick_params(labelsize=5)
    ax.set_title(title, fontsize=6)


def full_polar(ax, delta, title):
    counts, edges = np.histogram(delta % 360, bins=12, range=(0, 360))
    ax.bar(np.deg2rad(edges[:-1] + 15), counts, width=np.deg2rad(30) * 0.95, color="k", lw=0)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 90)))
    ax.yaxis.set_major_locator(plt.MaxNLocator(2))
    ax.tick_params(labelsize=5)
    ax.set_title(title, fontsize=6)


conv0, conv1 = list(CONVENTIONS)
fig = plt.figure(figsize=(18 * CM, 12 * CM))
axes = {k: fig.add_subplot(2, 3, i + 1, projection=None if k == "D" else "polar") for i, k in enumerate("ABCDE")}
for k, (conv, sign) in zip("AB", CONVENTIONS.items()):
    t = testsA[("A1 flank axis", conv)]
    half_polar(axes[k], rf_shape.axial_diff(selA.axis, sign * selA.pref_ori), selA.oriented_z,
               f"Flank axis - pref. orientation\n{conv}\n{fmt(t)}")
t = testsA[("A2 centre long axis", conv0)]
half_polar(axes["C"], rf_shape.axial_diff(selA.elong_axis + 90, selA.pref_ori), selA.elongation,
           f"Centre long axis + 90 - pref. orientation\n{conv0}\n{fmt(t)}")
sft = flk[flk.sftf_tuned]
axes["D"].scatter(sft.asym, sft.fit_dsi, s=5, color="k", alpha=0.6, linewidths=0)
st = testsB["asym vs fit_dsi (SF/TF tuned)"]
axes["D"].set_title(f"n = {st['n']}, ρ = {st['rho']:.2f} [{st['ci'][0]:.2f}, {st['ci'][1]:.2f}], "
                    f"p = {st['p']:.2g}", fontsize=6)
axes["D"].set_xlabel("RF asymmetry")
axes["D"].set_ylabel("Direction selectivity (fit)")
axes["D"].spines[["top", "right"]].set_visible(False)
full_polar(axes["E"], selB3.motion_dir - selB3.asym_dir,
           f"Pref. motion - asymmetry direction\n{conv0}\n{fmt(testsB3[conv0])}")
axes["F"] = fig.add_subplot(2, 3, 6)
axes["F"].scatter(selB4.rf_azi, np.cos(np.deg2rad(selB4.asym_dir)) * selB4.asym, s=5, color="k", alpha=0.6,
                  linewidths=0)
r = position[("rf_azi", "cos")]
axes["F"].set_title(f"n = {len(selB4)}, ρ(azimuth, cos) = {r[0]:.2f}, p = {r[1]:.2g}", fontsize=6)
axes["F"].set_xlabel("RF azimuth (deg)")
axes["F"].set_ylabel("Asymmetry along azimuth")
axes["F"].axhline(0, color="#8a8984", lw=0.8, ls="--")
axes["F"].spines[["top", "right"]].set_visible(False)
for letter, ax in axes.items():
    ax.text(-0.15, 1.12, letter, transform=ax.transAxes, fontweight="bold", fontsize=9, va="bottom")
fig.tight_layout()
style.savefig(SAVE_ROOT / "fig_rf_sftf_summary.pdf", fig=fig, bbox_inches="tight")

In [ ]:
# null distributions of the A1 / B3 permutation tests
fig, axes = plt.subplots(1, 4, figsize=(18 * CM, 4 * CM))
panels = [(f"A1 {c}", testsA[("A1 flank axis", c)]) for c in CONVENTIONS] + [
    (f"B3 {c}", testsB3[c]) for c in CONVENTIONS]
for ax, (name, t) in zip(axes, panels):
    ax.hist(t["null"], bins=50, color="#8a8984")
    ax.axvline(t["stat"], color="r")
    ax.set_title(f"{name}\np = {t['p']:.3g}", fontsize=6)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
cells.drop(columns=[c for c in cells.columns if c.endswith(("_h0", "_h1"))]).to_csv(
    SAVE_ROOT / "rf_shape_cells.csv", index=False)